# XGBoost with injury news and situational features

Does the boosting model improve with official injury reports and the situational features below? Features come
from `scripts/situational_features.py` (every value uses earlier ET dates only; `tests/test_situational_features.py`
checks this), injury reports from `scripts/collect_injuries.py` (report time at or before tipoff − 60 min).

| Family | What it measures | Source |
| --- | --- | --- |
| baseline (252) | the published model's features: win rates, rest, box-score windows, rotation shape | shared dataset |
| recency | exponentially weighted (half-life 5 games) net/offensive/defensive rating, plain and opponent-adjusted; schedule strength | team box scores |
| fatigue | games in last 4/7 days, miles since last game and in last 7 days, time-zone shift, altitude visit on ≤1 day rest, composite score | schedule + arena coordinates |
| rematch | same opponent in the previous game within 4 days, and that game's margin | schedule |
| shooting | 3PT% and eFG% "luck" (last 5 minus season), for and against; 3PT% shrunk toward the league | team box scores |
| lineup | rotation value (game score per minute × expected minutes, top 9) and the share available, judged by who played the previous game | player box scores |
| injury | players listed Out / Doubtful / Questionable on the official report (G League excluded) | NBA injury reports, Dec 2018 onward |
| notes | the same status counts read from the injury/status section of a team's own pregame game notes | game notes (selected teams and seasons; ~11% of TRAIN games) |

**Not possible with this data:** expected eFG% from shot location and defender distance needs tracking data, and
on/off net ratings need play-by-play. The `shooting` and `lineup` families are the box-score substitutes.

**Protocol:** fit on TRAIN (2012–13 to 2023–24), compare on DEV (2024–25). TEST (2025–26) is **not scored** here:
the published baseline already used its one TEST look, and scoring a new model on TEST is a decision to make
deliberately once the DEV comparison is settled. Model: XGBoost `hist`, averaged over 3 seeds; settings chosen on
DEV for the baseline features and then held fixed for every feature set, so any gain is not from extra tuning.

### How to run

```bash
uv pip install --python .venv/bin/python --group local --group notebook --group tabpfn
# Official injury reports (public NBA PDFs, Dec 2018 onward; cached; a few hours, three ranges in parallel):
cd scripts
../.venv/bin/python collect_injuries.py --splits train --first-season 2018 --last-season 2020 --output part_a.jsonl &
../.venv/bin/python collect_injuries.py --splits train --first-season 2021 --last-season 2023 --output part_b.jsonl &
../.venv/bin/python collect_injuries.py --splits dev test --output part_c.jsonl &
wait
```

Without the injury files the notebook still runs; the injury columns are then empty. The collector also fetches TEST
reports so they exist for a later TEST evaluation; this notebook never reads TEST rows.

In [ ]:
import json, sys, time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().resolve()
while not (ROOT / "config/gradient_boosting.json").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import run_gradient_boosting as rgb
import situational_features as sf
from collect_polymarket import load_price_observations
from evaluate_orthogonality import statistics, day_folds, logit
from nba_dataset import Dataset

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})
SEED = 230

def table(rows, columns, formats=None):
    formats = formats or {}
    out = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    for row in rows:
        out.append("| " + " | ".join("" if row[c] is None else format(row[c], formats.get(c, "")) for c in columns) + " |")
    display(Markdown("\n".join(out)))

started = time.time()
dataset = Dataset(ROOT)
inputs = dataset.inputs(("train", "dev"))                 # TEST rows are not loaded
labels = dataset.labels(("train", "dev"))
X_base, y = rgb.arrays(inputs, labels)
ids = np.array([r["game_id"] for r in inputs])
split = np.array([r["split"] for r in inputs])
season = np.array([r["season"] for r in inputs])
dates = np.array([str(r["game_date_et"]) for r in inputs])
TRAIN, DEV = split == "train", split == "dev"
history = {g["game_id"]: g for g in dataset.history()}
margin = np.array([history[g]["home_score"] - history[g]["away_score"] for g in ids], float)

injury_files = sorted((ROOT / "data/processed/injury_full").glob("part_*.jsonl"))
features = sf.from_dataset(dataset, injury_files, note_splits=("train", "dev"))
X_sit = np.array([[features[g][n] for n in sf.FEATURE_NAMES] for g in ids], float)
SIT = {n: j for j, n in enumerate(sf.FEATURE_NAMES)}
def columns(families):
    return [SIT[n] for n in sf.feature_names(families)]
stored = {r["game_id"]: r["p_boost"] for r in dataset.predictions("gradient_boosting")}
p_published = np.array([stored.get(g, np.nan) for g in ids])
print(f"{len(ids):,} TRAIN+DEV games, {X_base.shape[1]} baseline + {X_sit.shape[1]} new columns, built in {time.time() - started:.0f}s")
print(f"Injury files: {[f.name for f in injury_files]}")

## Injury-report coverage

In [ ]:
found = X_sit[:, SIT["home_injury_report_found"]] == 1
rows = [{"season": rgb.season_label(int(s)), "split": split[season == s][0].upper(), "games": int((season == s).sum()),
         "with_report": int((found & (season == s)).sum()), "share": (found & (season == s)).sum() / (season == s).sum()}
        for s in sorted(set(season)) if s >= 2018]
table(rows, ["season", "split", "games", "with_report", "share"], {"share": ".0%"})
listed = {k: X_sit[found, SIT[f"home_injury_{k}"]].mean() for k in ("out", "doubtful", "questionable")}
print("Average home-team listings per report:", {k: round(v, 2) for k, v in listed.items()})
noted = (X_sit[:, SIT["home_note_found"]] + X_sit[:, SIT["away_note_found"]]) > 0
print(f"Games with a game note: TRAIN {(noted & TRAIN).sum():,} of {TRAIN.sum():,}, DEV {(noted & DEV).sum():,} of {DEV.sum():,}")

## Settings chosen on DEV (baseline features only)

In [ ]:
def fit_predict(cols_base, cols_sit, params, objective="binary:logistic", target=None, fit_mask=TRAIN, predict_mask=DEV, seeds=(0, 1, 2)):
    X = np.hstack([X_base[:, cols_base], X_sit[:, cols_sit]])
    target = y if target is None else target
    out = []
    for s in seeds:
        kw = dict(tree_method="hist", objective=objective, random_state=SEED + s, n_jobs=16, **params)
        model = (xgb.XGBClassifier if objective == "binary:logistic" else xgb.XGBRegressor)(**kw)
        model.fit(X[fit_mask], target[fit_mask])
        out.append(model.predict_proba(X[predict_mask])[:, 1] if objective == "binary:logistic" else model.predict(X[predict_mask]))
    return np.mean(out, axis=0), model

BASE = list(range(X_base.shape[1]))
COMMON = dict(subsample=.8, colsample_bytree=.6, min_child_weight=20, reg_lambda=1.0)
grid = []
for depth in (2, 3, 4):
    for trees in (300, 600, 1000):
        params = dict(COMMON, max_depth=depth, n_estimators=trees, learning_rate=.03)
        prob, _ = fit_predict(BASE, [], params)
        grid.append({"max_depth": depth, "trees": trees, "dev_brier": rgb.score_predictions(y[DEV], prob)["brier"], "params": params})
best = min(grid, key=lambda r: r["dev_brier"])
PARAMS = best["params"]
table(grid, ["max_depth", "trees", "dev_brier"], {"dev_brier": ".4f"})
print(f"Selected: depth {best['max_depth']}, {best['trees']} trees (learning rate 0.03). "
      f"Published sklearn model on DEV: {rgb.score_predictions(y[DEV], p_published[DEV])['brier']:.4f}")

## Test A: loss functions

Win probability can come straight from a log-loss classifier, or from a point-margin regression converted to a
probability. The margin models use squared error, pseudo-Huber (slope 10 points: quadratic for small misses, linear
for blowouts) and absolute error. The margin-to-probability map is a logistic fit on **out-of-fold TRAIN** margins
(4 folds grouped by season), so DEV is not used to calibrate.

In [ ]:
def margin_probability(objective, cols_sit, extra=None):
    params = dict(PARAMS, **(extra or {}))
    groups = season[TRAIN]
    folds = {s: i % 4 for i, s in enumerate(sorted(set(groups)))}
    fold = np.array([folds[s] for s in groups])
    train_idx = np.flatnonzero(TRAIN)
    oof = np.zeros(TRAIN.sum())
    for f in range(4):
        fit_mask = np.zeros(len(y), bool); fit_mask[train_idx[fold != f]] = True
        pred_mask = np.zeros(len(y), bool); pred_mask[train_idx[fold == f]] = True
        oof[fold == f], _ = fit_predict(BASE, cols_sit, params, objective, margin, fit_mask, pred_mask, seeds=(0,))
    calibrator = LogisticRegression(C=1e6).fit(oof[:, None], y[TRAIN])
    dev_margin, _ = fit_predict(BASE, cols_sit, params, objective, margin)
    return calibrator.predict_proba(dev_margin[:, None])[:, 1], dev_margin

loss_rows, loss_forecasts = [], {}
for cols_name, cols_sit in (("baseline features", []), ("baseline + situational", columns(["recency", "fatigue", "rematch", "shooting", "lineup"]))):
    probs = {"log loss (classifier)": fit_predict(BASE, cols_sit, PARAMS)[0]}
    margin_mae = {}
    for label, objective, extra in (("squared error on margin", "reg:squarederror", None),
                                    ("pseudo-Huber on margin", "reg:pseudohubererror", {"huber_slope": 10.0, "base_score": float(np.median(margin[TRAIN]))}),
                                    ("absolute error on margin", "reg:absoluteerror", None)):
        probs[label], dev_margin = margin_probability(objective, cols_sit, extra)
        margin_mae[label] = float(np.mean(np.abs(dev_margin - margin[DEV])))
    boot = rgb.bootstrap_brier(y[DEV], probs, dates[DEV], 2000, SEED, "log loss (classifier)")
    for k, prob in probs.items():
        m = rgb.score_predictions(y[DEV], prob)
        diff = boot["difference_vs_reference"].get(k)
        loss_rows.append({"features": cols_name, "objective": k, "dev_brier": m["brier"], "log_loss": m["log_loss"], "roc_auc": m["roc_auc"],
                          "margin_mae": margin_mae.get(k), "vs_classifier": "" if diff is None else f"[{diff[0]:+.4f}, {diff[1]:+.4f}]"})
table(loss_rows, ["features", "objective", "dev_brier", "log_loss", "roc_auc", "margin_mae", "vs_classifier"],
      {"dev_brier": ".4f", "log_loss": ".4f", "roc_auc": ".3f", "margin_mae": ".2f"})
print("vs_classifier: 95% interval of Brier(objective) - Brier(log-loss classifier); negative = better.")

## Test B: which feature families help?

Log-loss classifier with the fixed settings. Each row adds one family to the 252 baseline features; the interval is
the paired day-clustered bootstrap of the DEV Brier difference against the baseline-features model.

In [ ]:
sets = {"baseline (252)": []}
for family in ("recency", "fatigue", "rematch", "shooting", "lineup", "injury", "notes"):
    sets[f"+ {family}"] = columns([family])
sets["+ all except injury"] = columns(["recency", "fatigue", "rematch", "shooting", "lineup"])
sets["+ all"] = columns(list(sf.FAMILIES))
forecasts, models = {}, {}
for name, cols_sit in sets.items():
    forecasts[name], models[name] = fit_predict(BASE, cols_sit, PARAMS)
boot = rgb.bootstrap_brier(y[DEV], forecasts, dates[DEV], 2000, SEED, "baseline (252)")
rows = []
for name, prob in forecasts.items():
    m = rgb.score_predictions(y[DEV], prob)
    diff = boot["difference_vs_reference"].get(name)
    rows.append({"features": name, "new_columns": len(sets[name]), "dev_brier": m["brier"], "log_loss": m["log_loss"],
                 "roc_auc": m["roc_auc"], "accuracy": m["accuracy"],
                 "vs_baseline": "" if diff is None else f"[{diff[0]:+.4f}, {diff[1]:+.4f}]"})
table(rows, ["features", "new_columns", "dev_brier", "log_loss", "roc_auc", "accuracy", "vs_baseline"],
      {"dev_brier": ".4f", "log_loss": ".4f", "roc_auc": ".3f", "accuracy": ".1%"})
FINAL = "+ all"

## Test C: injury reports where they exist

In [ ]:
has = DEV & found
rows = []
for name in ("baseline (252)", "+ all except injury", "+ all"):
    prob = forecasts[name][has[DEV]]
    rows.append({"forecast": name, "games": int(has.sum()), "dev_brier": rgb.score_predictions(y[has], prob)["brier"]})
table(rows, ["forecast", "games", "dev_brier"], {"dev_brier": ".4f"})
if has.sum() > 50:
    sub = {n: forecasts[n][has[DEV]] for n in ("+ all except injury", "+ all")}
    b = rgb.bootstrap_brier(y[has], sub, dates[has], 2000, SEED, "+ all except injury")
    lo, hi = b["difference_vs_reference"]["+ all"]
    print(f"Adding injury reports, DEV games with a report: Brier difference [{lo:+.4f}, {hi:+.4f}]")
out_diff = X_sit[:, SIT["diff_injury_out"]]
m = DEV & found
for label, sel in (("home has more players Out", m & (out_diff > 0)), ("equal", m & (out_diff == 0)), ("away has more Out", m & (out_diff < 0))):
    if sel.any():
        print(f"{label:28s} {sel.sum():4d} games, home won {y[sel].mean():.1%}")

## Test D: what the model uses

In [ ]:
names = list(rgb.FEATURE_NAMES) + sf.feature_names(list(sf.FAMILIES))
gain = models[FINAL].get_booster().get_score(importance_type="total_gain")
total = sum(gain.values())
imp = sorted(((names[int(k[1:])], v / total) for k, v in gain.items()), key=lambda kv: -kv[1])
fig, ax = plt.subplots(figsize=(8, 7))
top = imp[:25][::-1]
new = set(sf.FEATURE_NAMES)
ax.barh([n for n, _ in top], [v for _, v in top], color=["#DD8452" if n in new else "#4C72B0" for n, _ in top])
ax.set(xlabel="share of total gain", title="Top 25 features (orange = new)"); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()
family_of = {n: f for f, members in sf.FAMILIES.items() for side in ("home", "away", "diff") for n in [f"{side}_{m}" for m in members]}
share = Counter()
for n, v in imp:
    share[family_of.get(n, "baseline")] += v
table([{"family": f, "gain_share": v} for f, v in share.most_common()], ["family", "gain_share"], {"gain_share": ".1%"})

## Test E: TabPFN with the same feature sets

TabPFN-2.5 (pinned checkpoint and settings in `config/tabpfn.json`) gets all TRAIN games as context and predicts
DEV in one pass; nothing is trained. It runs on the configured device, `mps` (Apple GPU), where one fit takes about two
minutes. Elsewhere this section is skipped unless `TABPFN_DEVICE` names an available device (on `cpu` also set
`TABPFN_ALLOW_CPU_LARGE_DATASET=1`; expect hours). Needs the `tabpfn` dependency group.

In [ ]:
import os
tab_forecasts = {}
tab_config = json.loads((ROOT / "config/tabpfn.json").read_text())
device = os.environ.get("TABPFN_DEVICE", tab_config["model"]["device"])
try:
    import torch
    import run_tabpfn as rt
    available = {"mps": torch.backends.mps.is_available(), "cuda": torch.cuda.is_available(), "cpu": True}[device]
except ImportError:
    available = False
if not available or (device == "cpu" and not os.environ.get("TABPFN_ALLOW_CPU_LARGE_DATASET")):
    print(f"TabPFN skipped: device '{device}' unavailable or tabpfn group not installed.")
else:
    tab_config["model"]["device"] = device
    rt.download_model(tab_config)
    tab_sets = {"baseline (252)": [], "+ lineup": columns(["lineup"]),
                "+ all except injury": columns(["recency", "fatigue", "rematch", "shooting", "lineup"]), "+ all": columns(list(sf.FAMILIES))}
    for name, cols_sit in tab_sets.items():
        t = time.time()
        M = np.hstack([X_base, X_sit[:, cols_sit]])
        model = rt.model_for(tab_config)
        model.fit(M[TRAIN], y[TRAIN])
        tab_forecasts[name] = rt.positive_probabilities(model, M[DEV])
        print(f"TabPFN {name}: {time.time() - t:.0f}s", flush=True)
    b = rgb.bootstrap_brier(y[DEV], tab_forecasts, dates[DEV], 2000, SEED, "baseline (252)")
    rows = []
    for name, prob in tab_forecasts.items():
        m = rgb.score_predictions(y[DEV], prob)
        diff = b["difference_vs_reference"].get(name)
        rows.append({"features": name, "tabpfn_brier": m["brier"], "xgboost_brier": rgb.score_predictions(y[DEV], forecasts[name])["brier"],
                     "log_loss": m["log_loss"], "roc_auc": m["roc_auc"], "accuracy": m["accuracy"],
                     "vs_tabpfn_baseline": "" if diff is None else f"[{diff[0]:+.4f}, {diff[1]:+.4f}]"})
    table(rows, ["features", "tabpfn_brier", "xgboost_brier", "log_loss", "roc_auc", "accuracy", "vs_tabpfn_baseline"],
          {"tabpfn_brier": ".4f", "xgboost_brier": ".4f", "log_loss": ".4f", "roc_auc": ".3f", "accuracy": ".1%"})
    recorded = {r["game_id"]: r["p_home_win"] for r in dataset.table("predictions") if r["model_id"] == "tabpfn"}
    rec = np.array([recorded.get(g, np.nan) for g in ids[DEV]])
    if np.isfinite(rec).all():
        print(f"Published TabPFN DEV Brier {rgb.score_predictions(y[DEV], rec)['brier']:.4f}; "
              f"max |this run - published| = {np.abs(tab_forecasts['baseline (252)'] - rec).max():.4f}")

## Test F: against Polymarket on DEV

Market price at the model's cutoff (tipoff − 60 min). Orthogonality as in `scripts/evaluate_orthogonality.py`:
*does one forecast add to the other?* (coefficient on the log-odds difference, out-of-fold Brier gain).

In [ ]:
price = {r["game_id"]: r["home_probability"] for r in load_price_observations()
         if r["record_kind"] == "cutoff" and r["home_probability"] is not None}
q = np.array([price.get(g, np.nan) for g in ids])
priced = DEV & np.isfinite(q)
dv = priced[DEV]
cands = {"Polymarket": q[priced], "published sklearn model": p_published[priced],
         "XGBoost baseline features": forecasts["baseline (252)"][dv], "XGBoost + all new features": forecasts[FINAL][dv]}
cands.update({f"TabPFN {name}": prob[dv] for name, prob in tab_forecasts.items()})
b = rgb.bootstrap_brier(y[priced], cands, dates[priced], 2000, SEED, "Polymarket")
rows = []
for name, prob in cands.items():
    diff = b["difference_vs_reference"].get(name)
    rows.append({"forecast": name, "games": int(priced.sum()), "dev_brier": rgb.score_predictions(y[priced], prob)["brier"],
                 "minus_market": "" if diff is None else f"[{diff[0]:+.4f}, {diff[1]:+.4f}]"})
table(rows, ["forecast", "games", "dev_brier", "minus_market"], {"dev_brier": ".4f"})
folds = day_folds(dates[priced], 5, SEED)
rows = []
for name in ("XGBoost baseline features", "XGBoost + all new features"):
    for direction, base, other in (("market adds to model", cands[name], q[priced]), ("model adds to market", q[priced], cands[name])):
        s = statistics(base, other, y[priced], folds)
        rows.append({"model": name, "direction": direction, "logit_corr": s["logit_corr"], "delta_coef": s["delta_coef"],
                     "lr_p": s["lr_p"], "oof_brier_gain": s["oof_brier_gain"]})
table(rows, ["model", "direction", "logit_corr", "delta_coef", "lr_p", "oof_brier_gain"],
      {"logit_corr": ".3f", "delta_coef": "+.3f", "lr_p": ".2g", "oof_brier_gain": "+.4f"})

## Summary

_To be written after a full run (all injury reports, TabPFN on the Apple GPU)._

Preliminary, from a Windows run on 2026-11-10 with injury reports for only ~10% of recent games and no TabPFN
(XGBoost, DEV 2024–25):

- Baseline features: DEV Brier 0.2095 (published sklearn model 0.2103).
- Only the **lineup** family helped on its own: 0.2070, paired interval [−0.0046, −0.0005]. Recency, fatigue,
  rematch and shooting were within ±0.0005 of baseline; rematch slightly worse.
- All new features: 0.2067 ([−0.0053, −0.0005] vs baseline).
- Margin models (squared, pseudo-Huber, absolute error) were 0.0003–0.0008 better than the log-loss classifier;
  every interval included zero.
- Polymarket on the same DEV games: 0.1990. The market still adds to the model; the model adds nothing to the market.
- Game-note injury counts (1,472 TRAIN and 182 DEV games with a note) did not lower DEV Brier: adding them changed
  it by −0.0003 to +0.0004 for every feature set, all intervals including zero, also on the 182 DEV games with
  notes. The notes average ~0.6 "out" mentions per team (official reports ~2.4); many say "TBD" or link to Twitter.